# AB2 · A compare-any-two-groups tool (prototype)

> **Goal:** a function that lets a user pick two groups of countries and get an honest answer to "do firms in A earn different ROA than firms in B?", with the safeguards that a naive A/B calculator would lack. This notebook builds and stress-tests it; `src/analysis/compare.py` holds the code so the dashboard can call it later (nothing is wired into the dashboard yet).

| | |
|---|---|
| **What the tool returns** | The B-minus-A gap with a 95% interval, a wild cluster bootstrap p-value, a country-level permutation p-value, group sizes and automatic warnings. |
| **Why not a plain t-test on firms** | Firms in one country are not independent: all 400 firms of a country share its conditions. A firm-level t-test treats 10,000 firms as 10,000 independent draws and finds "significant" differences that are really 34 countries' worth of evidence. Section 3 measures how bad this is. |
| **Why not let users try many splits** | Each extra comparison is another lottery ticket. Section 4 shows how many "significant" splits appear by chance, and what the tool should do about it. |
| **Status** | A tool, not a hypothesis: no verdict and nothing is stored in `results/`. |

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils, compare
from utils import CONTROLS, fit_cluster, wild_cluster, bh

con = prep.connect()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile = country["dim"], country["profile"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
EU, NON_EU = list(dim.index[dim.is_eu]), list(dim.index[~dim.is_eu])
print(f"{len(dim)} countries ({len(EU)} EU, {len(NON_EU)} non-EU) | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

34 countries (21 EU, 13 non-EU) | 10,220 Yahoo company-years of 2,590 companies


## 1 · The tool in use
Three example comparisons, each B minus A on ROA (percentage points), controlling for sector, size and year.

In [2]:
groups = {
 "EU vs non-EU": (NON_EU, EU),
 "euro area vs other EU": (list(dim.index[dim.is_eu & ~dim.is_euro]), list(dim.index[dim.is_euro])),
 "Eastern vs Western EU": (list(dim.index[dim.is_eu & (dim.eu_group == "Eastern")]), list(dim.index[dim.is_eu & (dim.eu_group == "Western")])),
 "OECD vs non-OECD": (list(dim.index[~dim.is_oecd]), list(dim.index[dim.is_oecd])),
}
rows = []
for label, (a, b) in groups.items():
    try:
        r = compare.compare_groups(fy, a, b, B=999)
        rows.append({"comparison (B minus A)": label, "A": r["countries_a"], "B": r["countries_b"], "diff (pp)": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "p (perm)": r["p_perm"], "warnings": "; ".join(r["warnings"])})
    except ValueError as e:
        rows.append({"comparison (B minus A)": label, "warnings": f"REFUSED: {e}"})
pd.DataFrame(rows).set_index("comparison (B minus A)").round(4)

,A,B,diff (pp),ci_low,ci_high,p (wild),p (perm),warnings
comparison (B minus A),,,,,,,,
EU vs non-EU,13,21,-2.284,-3.412,-1.155,0.001,0.001,a country with fewer than 50 firm-years is inc...
euro area vs other EU,6,15,-0.776,-2.210,0.657,0.386,0.268,a group has fewer than 8 countries: the bootst...
Eastern vs Western EU,7,7,-2.373,-3.541,-1.205,0.006,0.058,a group has fewer than 8 countries: the bootst...
OECD vs non-OECD,7,27,-1.633,-3.204,-0.062,0.065,0.034,a group has fewer than 8 countries: the bootst...


**Guard rails built in:** overlapping groups are rejected; fewer than 3 countries per group is refused; fewer than 8 per group, very unbalanced groups, or a country with under 50 firm-years add a warning. Try it:

In [3]:
for a, b in [(["USA", "CAN"], ["DEU", "FRA", "ITA"]), (["USA", "CHN", "JPN"], ["USA", "KOR", "IND"])]:
    try:
        print(compare.compare_groups(fy, a, b)["warnings"])
    except ValueError as e:
        print("Refused:", e)

Refused: each group needs at least 3 countries with firm data (got 2 and 3)
Refused: groups overlap: ['USA']


## 2 · Filters: does the answer depend on which firms are included?
The same comparison (EU vs non-EU) with a sector filter and a size filter, as a dashboard user might apply them. Each filter changes the question being asked, so the result is shown with the number of firm-years behind it.

In [4]:
rows = []
for label, d in [("all firms", fy), ("industrials", fy[fy.sector == "Industrials"]), ("technology", fy[fy.sector == "Technology"]), ("financial services", fy[fy.sector == "Financial Services"]),
                 ("smallest third in each country", fy[fy.size_rank <= 1/3]), ("largest third in each country", fy[fy.size_rank > 2/3])]:
    ctl = "C(fiscal_year)" + (" + C(sector)" if label in ("all firms",) or "third" in label else "") + (" + size_rank" if "third" not in label else "")
    try:
        r = compare.compare_groups(d, NON_EU, EU, B=499, controls=ctl)
        rows.append({"firms": label, "diff (pp)": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "p (perm)": r["p_perm"], "firm-years": r["firm_years_a"] + r["firm_years_b"], "countries": r["countries_a"] + r["countries_b"]})
    except ValueError as e:
        rows.append({"firms": label, "diff (pp)": np.nan})
flt = pd.DataFrame(rows).set_index("firms"); flt.round(3)

,diff (pp),ci_low,ci_high,p (wild),p (perm),firm-years,countries
firms,,,,,,,
all firms,-2.284,-3.412,-1.155,0.002,0.001,10220,34
industrials,-0.715,-2.078,0.647,0.340,0.299,1240,33
technology,-3.004,-6.536,0.528,0.152,0.136,841,30
financial services,-1.179,-2.781,0.422,0.224,0.241,1169,33
smallest third in each country,-5.360,-7.877,-2.844,0.002,0.000,3373,34
largest third in each country,-0.151,-0.878,0.576,0.702,0.646,3445,34


## 3 · Why the tool clusters by country: a placebo experiment
Assign the 34 countries to two groups **at random** (17 vs 17) many times. There is no real difference by construction, so a valid test should call 5% of them "significant". We compare the naive firm-level t-test with the tool's two country-aware p-values.

In [5]:
rng = np.random.default_rng(42)
d = fy.dropna(subset=["roa_pp"]); iso = np.array(sorted(d.iso3.unique()))
cmeans = compare._country_means(d, "roa_pp", CONTROLS)
N = 400
naive, perm = [], []
for _ in range(N):
    b = set(rng.choice(iso, 17, replace=False)); a = set(iso) - b
    isb = d.iso3.isin(b).values
    naive.append(stats.ttest_ind(d.roa_pp[isb], d.roa_pp[~isb], equal_var=False).pvalue)
    ma, mb = cmeans[cmeans.index.isin(a)], cmeans[cmeans.index.isin(b)]
    perm.append(stats.ttest_ind(mb, ma, equal_var=False).pvalue)       # country-level Welch t, fast stand-in for the permutation p
# a small number of full tool runs (wild bootstrap) as a check
wild = [compare.compare_groups(d, list(set(iso) - set(bb)), list(bb), B=299, perm=False)["p_wild"] for bb in (set(rng.choice(iso, 17, replace=False)) for _ in range(60))]
res = pd.Series({"firm-level t-test (naive)": np.mean(np.array(naive) < 0.05), "country-level t-test": np.mean(np.array(perm) < 0.05), "tool's wild cluster bootstrap (60 runs)": np.mean(np.array(wild) < 0.05)})
fig = px.bar(res.reset_index(), x="index", y=0, title=f"Share of random splits called 'significant' at 5% (should be 5%; {N} splits)", labels={"index": "", "0": "false positive rate"})
fig.add_hline(y=0.05, line_dash="dot", line_color="black"); fig.update_layout(height=340, yaxis_tickformat=".0%"); fig.show()
print((res * 100).round(1).astype(str).add("%").to_string())

firm-level t-test (naive)                  64.5%
country-level t-test                        3.2%
tool's wild cluster bootstrap (60 runs)     6.7%


## 4 · The many-splits problem
Split the 34 countries at the median of each of the ~30 country features (high vs low) and test every split. If a user can try them all and report the best one, how many "discoveries" are there?

In [6]:
rows = []
for f in country["feat_cols"]:
    lo, hi = compare.median_split(profile[f])
    r = compare.compare_groups(fy, lo, hi, B=2, perm=True)
    rows.append({"feature": f, "diff (pp)": r["diff"], "p (perm)": r["p_perm"], "countries lo/hi": f"{r['countries_a']}/{r['countries_b']}"})
scan = pd.DataFrame(rows).set_index("feature").sort_values("p (perm)"); scan["q (BH)"] = bh(scan["p (perm)"])
n_raw, n_bh = int((scan["p (perm)"] < 0.05).sum()), int((scan["q (BH)"] < 0.10).sum())
print(f"{len(scan)} median splits tested: {n_raw} have p < 0.05 (about {0.05 * len(scan):.1f} expected by chance alone); {n_bh} pass Benjamini-Hochberg q < 0.10.")
print("Note: these features are strongly correlated with each other (development), so the splits are not independent tests; the correction is, if anything, too lenient.")
# placebo benchmark: shuffle which country has which ROA, so no feature can truly matter, and count the splits that still reach p < 0.05
rng = np.random.default_rng(7)
cm_all = compare._country_means(fy.dropna(subset=["roa_pp"]), "roa_pp", CONTROLS)
splits = {f: compare.median_split(profile[f]) for f in country["feat_cols"]}
hits = []
for _ in range(100):
    shuf = pd.Series(rng.permutation(cm_all.values), index=cm_all.index)
    hits.append(sum(stats.ttest_ind(shuf[shuf.index.isin(hi)], shuf[shuf.index.isin(lo)], equal_var=False).pvalue < 0.05 for lo, hi in splits.values()))
print(f"Placebo (ROA shuffled across countries, 100 repetitions): on average {np.mean(hits):.1f} of {len(splits)} splits reach p < 0.05; in the worst repetition {max(hits)}. Real data: {n_raw}.")
scan.head(12).round(4)

35 median splits tested: 14 have p < 0.05 (about 1.8 expected by chance alone); 13 pass Benjamini-Hochberg q < 0.10.
Note: these features are strongly correlated with each other (development), so the splits are not independent tests; the correction is, if anything, too lenient.
Placebo (ROA shuffled across countries, 100 repetitions): on average 1.9 of 35 splits reach p < 0.05; in the worst repetition 9. Real data: 14.


,diff (pp),p (perm),countries lo/hi,q (BH)
feature,,,,
voice_accountability,-2.486,0.001,17/17,0.010
life_expectancy,-2.491,0.001,17/17,0.010
population,2.189,0.003,17/17,0.029
rule_of_law,-2.034,0.007,17/17,0.048
gini,2.118,0.008,13/13,0.048
gdp_per_capita_ppp,-1.992,0.008,17/17,0.048
pop_65plus,-1.967,0.013,17/17,0.058
trade_gdp,-1.804,0.013,17/17,0.058
gov_effectiveness,-1.655,0.020,17/17,0.076


The real data shows far more hits than the placebo, because all these features measure aspects of development, which genuinely goes with ROA (H0); the placebo line is what a user would face with an outcome that has no real link, where some split still looks 'significant' in many repetitions. **What the tool should do with this:** show a running count of comparisons made in the session ("this is comparison #7") and, when more than one has been tried, report the BH-adjusted q next to the p-value. A user looking at a single pre-chosen comparison should see p; a user browsing should see q.

## 5 · How large a gap can 34 countries detect?
A comparison with few countries can only detect large gaps. The minimum detectable difference (80% power, 5% level) for several group sizes, using the spread of country means (SD in pp, after sector, size and year).

In [7]:
sd = float(cmeans.std())
sizes = [(3, 3), (5, 5), (8, 8), (13, 21), (17, 17)]
mde = pd.DataFrame([{"group sizes (A, B)": f"{a}, {b}", "min detectable gap (pp ROA)": compare.min_detectable(sd, a, b)} for a, b in sizes]).set_index("group sizes (A, B)")
print(f"SD of country-mean ROA across the 34 countries: {sd:.2f} pp.")
mde.round(2)

SD of country-mean ROA across the 34 countries: 1.78 pp.


,min detectable gap (pp ROA)
"group sizes (A, B)",
"3, 3",4.060
"5, 5",3.140
"8, 8",2.490
"13, 21",1.750
"17, 17",1.710


## 6 · Specification for the dashboard (not built)
**Inputs:** two country selectors (A and B, with presets such as EU / non-EU, high / low development), optional sector and size filters, outcome fixed to ROA (percentage points).

**Outputs:** the B-minus-A gap with its interval, both p-values, the number of countries and firm-years in each group, the minimum detectable gap for those group sizes, the session comparison counter with the BH-adjusted q, and the warnings.

**Wording on the page:** "difference between groups", never "effect" or "A/B test result"; a fixed caption that groups are not randomly assigned.

**Cost:** `compare_groups` with B = 999 runs in about 0.3 seconds on the full sample, fast enough for a callback.

### Limits
- Controls are fixed (sector, size rank, year); a user cannot add others, so confounding by development or institutions is not adjusted unless a preset does it.
- Wild bootstrap with few clusters per group is unreliable, which is why the permutation p-value and the warnings exist.
- The placebo experiment covers random splits; real splits such as EU vs non-EU are correlated with development, so false-positive rates for them are higher.
- Association, not causation.

In [8]:
con.close()